# 01 — Universo de productos SEPA

Procesa un ZIP **diario** del SEPA e identifica todos los productos únicos (EANs),
su cobertura por cadena y provincia, y genera el resumen exportable a Excel.

**Input:** `data/input/diarios/AAAA-MM-DD.zip`  
**Output:** `data/output/productos_unicos_AAAA-MM-DD.xlsx`

In [ ]:
import sys, os
from pathlib import Path

# Detectar entorno
IS_COLAB = 'google.colab' in str(get_ipython()) if get_ipython() else False
if IS_COLAB:
    # Clonar o montar el repo en Colab
    PROJECT_ROOT = Path('/content/analisis_precios_minoristas_UADE')
else:
    PROJECT_ROOT = Path('..').resolve()

if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

print(f'Raíz del proyecto: {PROJECT_ROOT}')

In [ ]:
# ── CONFIGURACIÓN ──────────────────────────────────────────────────────────
# Modificar según el archivo a procesar
ZIP_DIARIO = PROJECT_ROOT / 'data' / 'input' / 'diarios' / '2026-04-26.zip'
MAESTRO_PRODUCTOS = PROJECT_ROOT / 'data' / 'masters' / 'Maestro de Productos Interno.xlsx'
MAESTRO_SUCURSALES = PROJECT_ROOT / 'data' / 'masters' / 'maestro_sucursales_completo.xlsx'

print(f'ZIP: {ZIP_DIARIO} — existe: {ZIP_DIARIO.exists()}')
print(f'Maestro productos: {MAESTRO_PRODUCTOS.exists()}')
print(f'Maestro sucursales: {MAESTRO_SUCURSALES.exists()}')

In [ ]:
from sepa.pipeline.loader import read_daily_zip, load_master_products, load_master_branches
from sepa.pipeline.cleaner import normalize_ean_column, filter_valid_coordinates
from sepa.pipeline.enricher import enrich_with_products, enrich_with_branches, filter_excluded_chains

# Leer ZIP
raw = read_daily_zip(ZIP_DIARIO)
print('Archivos encontrados:', list(raw.keys()))
for k, v in raw.items():
    print(f'  {k}: {len(v):,} filas, {v.shape[1]} columnas')

In [ ]:
import pandas as pd
from collections import defaultdict

df_prod = raw.get('productos', pd.DataFrame())
df_suc  = raw.get('sucursales', pd.DataFrame())
df_com  = raw.get('comercio', pd.DataFrame())

# Normalizar EAN
df_prod = normalize_ean_column(df_prod, 'productos_ean')
df_prod = filter_excluded_chains(df_prod)

print(f'Productos: {len(df_prod):,} registros, {df_prod["ean_norm"].nunique():,} EANs únicos')
print(f'Sucursales: {len(df_suc):,}  |  Comercios: {len(df_com):,}')

In [ ]:
# Enriquecer con maestros (opcional — si están disponibles)
if MAESTRO_PRODUCTOS.exists():
    mp = load_master_products(MAESTRO_PRODUCTOS)
    df_prod = enrich_with_products(df_prod, mp)
    print('Enriquecimiento con maestro de productos: OK')

if MAESTRO_SUCURSALES.exists():
    mb = load_master_branches(MAESTRO_SUCURSALES)
    # Unir sucursales con productos
    df_prod = enrich_with_branches(df_prod, mb)
    print('Enriquecimiento con maestro de sucursales: OK')

In [ ]:
# Tabla de productos únicos con métricas de cobertura
def build_product_summary(df):
    price_cols = [c for c in df.columns if c.startswith('productos_precio')]
    
    grp_cols = ['ean_norm']
    meta_cols = [c for c in ['descripcion', 'marca', 'rubro', 'categoria'] if c in df.columns]
    cover_cols = [c for c in ['provincia', 'cadena', 'id_sucursal'] if c in df.columns]
    
    rows = []
    for ean, g in df.groupby('ean_norm'):
        row = {'ean_norm': ean, 'n_registros': len(g)}
        for c in meta_cols:
            row[c] = g[c].mode().iloc[0] if not g[c].dropna().empty else None
        if 'provincia' in g.columns:
            row['n_provincias'] = g['provincia'].nunique()
        if 'cadena' in g.columns:
            row['n_cadenas'] = g['cadena'].nunique()
        if 'id_sucursal' in g.columns:
            row['n_sucursales'] = g['id_sucursal'].nunique()
        rows.append(row)
    
    return pd.DataFrame(rows).sort_values('n_registros', ascending=False)

df_summary = build_product_summary(df_prod)
print(f'Resumen generado: {len(df_summary):,} productos únicos')
df_summary.head(10)

In [ ]:
# Cobertura por provincia
if 'provincia' in df_prod.columns:
    cob_prov = df_prod.groupby('provincia').agg(
        n_productos=('ean_norm', 'nunique'),
        n_cadenas=('cadena', 'nunique') if 'cadena' in df_prod.columns else ('ean_norm', 'count'),
        n_registros=('ean_norm', 'count')
    ).reset_index().sort_values('n_productos', ascending=False)
    print('Cobertura por provincia:')
    display(cob_prov)
else:
    cob_prov = pd.DataFrame()
    print('Provincia no disponible (falta maestro de sucursales)')

In [ ]:
# Exportar a Excel
from sepa.viz.exports import save_excel

label = ZIP_DIARIO.stem
output_path = PROJECT_ROOT / 'data' / 'output' / f'productos_unicos_{label}.xlsx'

sheets = {
    'productos_unicos': df_summary,
    'cobertura_provincia': cob_prov,
}
save_excel(sheets, output_path)
print(f'Exportado: {output_path}')